## <small>
Copyright (c) 2017-21 Andrew Glassner

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
</small>



# Deep Learning: A Visual Approach
## by Andrew Glassner, https://glassner.com
### Order: https://nostarch.com/deep-learning-visual-approach
### GitHub: https://github.com/blueberrymusic
------

### What's in this notebook

This notebook is provided as a “behind-the-scenes” look at code used to make some of the figures in this chapter. It is cleaned up a bit from the original code that I hacked together, and is only lightly commented. I wrote the code to be easy to interpret and understand, even for those who are new to Python. I tried never to be clever or even more efficient at the cost of being harder to understand. The code is in Python3, using the versions of libraries as of April 2021.

This notebook may contain additional code to create models and images not in the book. That material is included here to demonstrate additional techniques.

Note that I've included the output cells in this saved notebook, but Jupyter doesn't save the variables or data that were used to generate them. To recreate any cell's output, evaluate all the cells from the start up to that cell. A convenient way to experiment is to first choose "Restart & Run All" from the Kernel menu, so that everything's been defined and is up to date. Then you can experiment using the variables, data, functions, and other stuff defined in this notebook.

## Chapter 19: RNNs - Notebook 2: Characters

Some code here is inspired by https://github.com/karpathy/char-rnn

The Holmes data can be found at Project Gutenberg
https://www.gutenberg.org/ebooks/search/?query=holmes

I combined three books of short stories into one big text file:

- “The Adventures of Sherlock Holmes by Arthur Conan Doyle”
- “The Return of Sherlock Holmes by Arthur Conan Doyle”
- “The Memoirs of Sherlock Holmes by Arthur Conan Doyle”

Note that due to the use of random numbers, your output will almost surely be different than mine.

In [20]:
from keras.models import Sequential
from keras.layers import Dense, Activation
from keras.layers import LSTM
from keras.optimizers import RMSprop
import numpy as np
import random
import sys

In [21]:
# Workaround for Keras issues on Mac computers (you can comment this
# out if you're not on a Mac, or not having problems)
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [22]:
# Make a File_Helper for saving and loading files.

save_files = False

try:
    import os, sys, inspect

    current_dir = os.path.dirname(os.path.abspath(inspect.getfile(inspect.currentframe())))
    sys.path.insert(0, os.path.dirname(current_dir))  # path to parent dir

    from DLBasics_Utilities import File_Helper
    print("Using real DLBasics_Utilities.File_Helper")

except ModuleNotFoundError:
    import os

    class File_Helper:
        def __init__(self, save_files=False):
            self.save_files = save_files
            self.INPUT_DIR = os.getcwd()

        def get_input_data_dir(self):
            d = self.INPUT_DIR
            print(f"get_input_data_dir (stub) -> {d}")
            return d

        def get_input_file_path(self, filename):
            path = os.path.join(self.INPUT_DIR, filename)
            print(f"get_input_file_path (stub) -> {path}")
            return path

        def save_figure(self, fig, filename, *args, **kwargs):
            if filename:
                print(f"save_figure (stub) – would save '{filename}', but skipping.")

        def load_model_weights(self, model, weights_filename):
            print(f"load_model_weights (stub) – no real weights file '{weights_filename}', returning False.")
            return False

        def save_model_weights(self, model, weights_filename):
            if self.save_files:
                model.save_weights(weights_filename)
                print(f"save_model_weights (stub) – saved weights to '{weights_filename}'")
            else:
                print(f"save_model_weights (stub) – would save '{weights_filename}', but skipping.")

        def __repr__(self):
            return f"<Stub File_Helper save_files={self.save_files}, INPUT_DIR='{self.INPUT_DIR}'>"

    print("Using stub File_Helper (DLBasics_Utilities not found)")

file_helper = File_Helper(save_files)
file_helper

Using stub File_Helper (DLBasics_Utilities not found)


<Stub File_Helper save_files=False, INPUT_DIR='/content'>

In [23]:
def get_text(input_file):
    # open the input file and do minor processing
    file = open(input_file, 'r', encoding='utf8')
    text = file.read()
    file.close()
    #text = text.lower()
    # replace newlines with blanks, and double blanks with singles
    text = text.replace('\n',' ')
    text = text.replace('  ', ' ')
    print('corpus length:', len(text))
    return text

In [24]:
def build_dictionaries(text):
    unique_chars = sorted(list(set(text)))
    print('total unique chars:', len(unique_chars))
    char_to_index = dict((ch, index) for index, ch in enumerate(unique_chars))
    index_to_char = dict((index, ch) for index, ch in enumerate(unique_chars))
    return (unique_chars, char_to_index, index_to_char)

In [25]:
def build_fragments(text, window_length):
    # make overlapping fragments of window_length characters
    fragments = []
    targets = []
    for i in range(0, len(text)-window_length, window_step):
        fragments.append(text[i: i + window_length])
        targets.append(text[i + window_length])
    print('number of fragments of length window_length=',window_length,':', len(fragments))
    return (fragments, targets)

In [26]:
def encode_training_data(fragments, window_length, targets, char_to_index, index_to_char):
    # Turn inputs and targets into one-hot versions
    X = np.zeros((len(fragments), window_length, len(char_to_index)), dtype=bool)
    y = np.zeros((len(fragments), len(char_to_index)), dtype=bool)
    for i, fragment in enumerate(fragments):
        for t, char in enumerate(fragment):
            X[i, t, char_to_index[char]] = 1
        y[i, char_to_index[targets[i]]] = 1
    return (X, y)

In [40]:
def build_model(window_length, num_unique_chars):
    # build the model. Two layers of a single LSTM cell with 128 elements of memory,
    # then a dense layer with as many outputs as there are characters (89)
    # We'll train with the RMSprop optimizer. Some experiments suggest that
    # a learning rate of 0.01 is a good place to start.
    model = Sequential()
    model.add(LSTM(128, return_sequences=True, input_shape=(window_length, num_unique_chars)))
    model.add(LSTM(128))
    model.add(Dense(num_unique_chars, activation='softmax'))
    optimizer = RMSprop(learning_rate=0.01) # Changed lr to learning_rate
    model.compile(loss='categorical_crossentropy', optimizer=optimizer)
    return model

In [28]:
# adjust our probabilities to add "heat"
def sample(preds, temperature=1.0):
    # helper function to sample an index from a probability array
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [29]:
# print a string to the screen and also save it in the file
def print_string(out_str='', file_writer=None):
    print(out_str, end='')
    if file_writer != None:
        file_writer.write(out_str)

In [30]:
def generate_text(model, X, y, number_of_epochs, temperatures, index_to_char, char_to_index, file_writer):
    # train the model, output generated text after each iteration
    for iteration in range(number_of_epochs):
        print_string('--------------------------------------------------\n', file_writer)
        print_string('Iteration '+str(iteration)+'\n', file_writer)
        history = model.fit(X, y, batch_size=batch_size, epochs=1)
        start_index = random.randint(0, len(text) - window_length - 1)

        for temperature in temperatures:
            print_string('\n----- temperature: '+str(temperature)+'\n', file_writer)
            sentence = text[start_index: start_index + window_length]
            generated = sentence
            print_string('----- Generating with seed: <'+sentence+'>\n', file_writer)

            for i in range(generated_text_length):
                x = np.zeros((1, window_length, len(index_to_char)))
                for t, char in enumerate(sentence):
                    x[0, t, char_to_index[char]] = 1.

                preds = model.predict(x, verbose=0)[0]
                next_index = sample(preds, temperature)
                next_char = index_to_char[next_index]

                generated += next_char
                sentence = sentence[1:] + next_char

            print_string(generated+'\n\n', file_writer)
            file_writer.flush()
    #print_string('\n', file_writer)

In [31]:
# set the globals
window_length = 40
window_step = 3
number_of_epochs = 100
generated_text_length = 1000
batch_size = 100
input_dir = file_helper.get_input_data_dir()

# Fix: The File_Helper stub does not have 'get_saved_output_dir' or 'check_for_directory'.
# We will define output_dir directly and ensure the directory exists using the 'os' module.
import os # Ensure os is imported
output_dir = os.path.join(input_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

input_file = input_dir+'/holmes.txt'
output_file =  output_dir+'/holmes-by-char.txt'
File_writer = open(output_file, 'w')

get_input_data_dir (stub) -> /content


In [38]:
import os, requests

url = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"  # Sherlock Holmes (Project Gutenberg)
dest = "/content/holmes.txt"

os.makedirs(os.path.dirname(dest), exist_ok=True)

print("Downloading holmes.txt...")
resp = requests.get(url)
resp.raise_for_status()
with open(dest, "w", encoding="utf8") as f:
    f.write(resp.text)
print("Saved to:", dest, "exists:", os.path.exists(dest))

input_file = dest  # make sure input_file points to the file

Saved to: /content/holmes.txt exists: True


In [41]:
# get text data structures, build the model

text = get_text(input_file)
unique_chars, char_to_index, index_to_char = build_dictionaries(text)
fragments, targets = build_fragments(text, window_length)
X, y = encode_training_data(fragments, window_length, targets, char_to_index, index_to_char)
model = build_model(window_length, len(char_to_index))

# Show the model we're using
model.summary()

corpus length: 578647
total unique chars: 98
number of fragments of length window_length= 40 : 192869


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 40, 128)        │       116,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 98)             │        12,642 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 260,450 (1017.38 KB)

 Trainable params: 260,450 (1017.38 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
number_of_epochs = 2
temperatures = [0.5, 1.0, 1.5]
generate_text(model, X, y, number_of_epochs, temperatures, index_to_char, char_to_index, File_writer)
# wrap up when we're done
File_writer.close()

--------------------------------------------------
Iteration 0
1929/1929 ━━━━━━━━━━━━━━━━━━━━ 407s 210ms/step - loss: 2.0494

----- temperature: 0.5
----- Generating with seed: < attentions. The dog is let loose at nig>
 attentions. The dog is let loose at night of the cournty of did that his can of the tore to it.” “You may compart of a sight in the could do in where we was only was a srepers of the little man her trrough a pordse of the of the man with the were of the ote to the best to he with the ground of the meanerse what he with of a sivery with the lough that I was a fould the moct that he need house with the conding a paners of the dead to my work, and not a forevered to to the like in the couns in the bore of the book to me to would be to it in the dese of the courried with the tair of the dister in the could in be in his ever of the wittle are with the what to be the seared your in in only perteaper. “Wat a very lided in his cater to me of the termesting of the could in the 